In [1]:
from collections.abc import Sequence
from dataclasses import dataclass
from time import perf_counter

import numpy as np

from multihawk import BaselineSpec, KernelSpec, simulate_hawkes


@dataclass(frozen=True)
class BenchmarkCase:
    """Definition of a single benchmark scenario."""

    name: str
    baseline: Sequence[float]
    alpha: Sequence[Sequence[float]]
    lambda_: Sequence[Sequence[float]]

    def run(self) -> None:
        """Execute the benchmark and record the elapsed wall-clock time."""
        baseline_spec = BaselineSpec(kind="constant", params={"values": list(self.baseline)})
        kernel_spec = KernelSpec(
            kind="exponential", params={"beta": [list(row) for row in self.lambda_]}
        )
        rng = np.random.default_rng(42)
        T = 5_000_000

        start = perf_counter()
        _ = simulate_hawkes(
            t_max=T,
            baseline=baseline_spec,
            alpha=[list(row) for row in self.alpha],
            kernel=kernel_spec,
            rng=rng,
        )
        duration = perf_counter() - start
        print(f"Elapsed time for {self.name}: {duration}")

In [2]:
# py-hawkes benchmark cases for exponential kernels
# See https://github.com/ragoragino/py-hawkes
def build_cases() -> list[BenchmarkCase]:
    """Create benchmark configurations mirroring py-hawkes exponential cases."""
    return [
        BenchmarkCase(
            name="1D exponential kernel with constant baseline",
            baseline=[0.5],
            alpha=[[0.4]],
            lambda_=[[0.8]],
        ),
        BenchmarkCase(
            name="2D exponential kernel with constant baseline",
            baseline=[0.25, 0.25],
            alpha=[[0.3, 0.4], [0.4, 0.5]],
            lambda_=[[1.0, 2.4], [1.5, 2.0]],
        ),
        BenchmarkCase(
            name="3D exponential kernel with constant baseline",
            baseline=[0.15, 0.15, 0.15],
            alpha=[[0.4, 0.3, 0.1], [0.3, 0.4, 0.23], [0.18, 0.43, 0.31]],
            lambda_=[[1.0, 2.0, 9.0], [1.2, 1.8, 1.4], [4.9, 1.15, 8.5]],
        ),
    ]

In [3]:
# About 2 times faster than py-hawkes benchmark on these 3 cases
cases = build_cases()
cases[0].run()
cases[1].run()
cases[2].run()

Elapsed time for 1D exponential kernel with constant baseline: 0.391899458001717
Elapsed time for 2D exponential kernel with constant baseline: 1.6722965840017423
Elapsed time for 3D exponential kernel with constant baseline: 2.5225302510079928
